In [10]:
# We will attempty to build a toy version of MIRA, following the paper's specifications and see if we can later tweak some of the 
# hyper parameters

![Screenshot](./views/graph1.png)

In [ ]:
# Let's start by importing the dataset as it is publicly available 

from huggingface_hub import snapshot_download

# since the train dataset is 7TB, we will only bring in two shards which is around 6GB

snapshot_download(
    "kyutai/rocket-science",
    repo_type='dataset',
    allow_patterns = ['test/index.json', 'test/dataset_00000.tar','test/dataset_00001.tar'], # the only 3 files we want (two shards)
    local_dir = './data/rocket'
)

# takes around 5 mins, you can give it a scroll

Fetching 3 files: 100%|██████████| 3/3 [05:40<00:00, 113.35s/it]


'/Users/achrafbayi/Desktop/MIRA/data/rocket'

In [ ]:
# now let's unpack the tar balls now

import os, tarfile

file1 = './data/rocket/test/dataset_00000.tar'
file2 = './data/rocket/test/dataset_00001.tar'
files = [file1, file2] # feel free to add as many shards as you need/can
dst = './data/rocket/test/unpacked'

os.makedirs(dst, exist_ok=True)

for file in files:
    with tarfile.open(file) as ball:
        ball.extractall(dst,filter='data')

# we now have the whole two shard dataset in disk ready to use

In [ ]:
# second step from the diagram above is to format it and feed it to the dinoV3-L feature extractor 
# (available on : https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m) request access asap to not have to sit
#  waiting for approval mid work

<img src="./views/graph2.png" width="600">

<b>Figure 3 Codec.</b> A frozen pretrained feature extractor (DINOv3-L) extracts per-frame features, mixed across several
intermediate layers, and a learned linear bottleneck downsamples them by 2 × 2 in space and 2× in time and projects
to a latent z with C channels. The decoder reverses these steps: a spatial upsampling restores the spatial resolution, a
causal spatio-temporal vision transformer reconstructs the video, and a temporal upsampling restores the frame rate.

(this is not figure 3 btw, but the figure 3 description explaines well what we need to do)

It is also quite important to understand that the Codec (encoder+decoder) are trained independently from the world model predictor and is frozen before predictor training time

In [3]:
# as seen above a shared encoder is used and then concats on all of the 4 frames outputs in latent space to get the full visual 
# latent space/embeddings (to which we'll later add the other signals such as physics, key bindings, ect)
# let's hence import our encoder from hugging-face and see wassup

import torch
import torch.nn as nn
from transformers import AutoModel

device = 'mps' if torch.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

# let's now define the encoder as a class that will implement all requirements from figure 3 and the graph which are : 
# (done individualy for all 4 files btw)
# 1) extract per frame features across several intermediate layers (exact mix is in table 9) using the DINOv3-L (torch.no_grad())
# 2) mix those embeddings by computing the mean, with no learnt bias (no NN involved)
# 3) linear bottle neck NN layer to down scale the obtained embeddings to our chosen channel number C (table 9) (Trained)
# 4) concat all 4 outputs

# from Table 9 of the implementation details, we get the exact codec training figures
# BatchSize is the number of frames we ingest

# moreover, Dino is a ViT style transformer, which hence splits our frames into patches and feeds them sequentially as tokens to the transformer
# and returns token embeddings

# moreover as for dimentionality, the paper mentions we need to preprocess our data into 288x512 format, which hence implies that the 
# that the number of tokens ingested and output-ed by the DINO ViT is of H/16 * W/16 = (288/16)*(512/16) = 18 * 32 ,
# hence the dimension of our VIT output is a tensor of shape (18, 32, 1024) and the multi-intermediate layers with the table 9 specification
# hence has us working with a (7, 18, 32, 1024) tensor that we scale down to a (18, 32, 1024) tensor through mixing
# and later, the t+1 merged with the other t frame which hence has us working with a (2, 18, 32, 1024) tensor is bottlenecked down
# through a linear neural net back to (9, 16, 32) embedding space (divided by 2x2 in space and 2x in time as per the paper)
# one interesting thing I might want to try out is using a neural net to mix rather than the mean (per the paper C = 32)

class Encoder(nn.Module):

    def __init__(self, layers=(11, 13, 15, 17, 19, 21, 23), C=32, BatchSize=40, H=288, W=512, n_embd=1024): # ugly number of layers
        super().__init__()
        self.dino = AutoModel.from_pretrained('facebook/dinov3-vitl16-pretrain-lvd1689m') # import DINO model
        self.dino.eval().requires_grad_(False) # making sure we don't backprop/optimize through dino (300M params)
        # btw for ViT-L, all embedding spaces have 1024 channels, hence : 
        self.layers = layers
        self.n_embd = n_embd
        self.C = C
        self.H = H
        self.W = W
        k = len(layers)
        self.mix = nn.ParameterList([nn.Parameter(torch.full((H//16, W//16, n_embd),1/k)) for _ in range(k)]) # mixing into a single embedding space
        # here we explicitely blow up by one dimension the self mix to allow a granualrity-based mix, with different weights for 
        # each layer, we want to be able to learn if layer 11, 17 or 23 need more gain at a specific cell of the 18*32 
        # matrix of embedding vectors which hence allows the encoder to learn for example to differentiate foreground/position from 
        # texture if those sit at different layers of the DINO encoding
        # btw here the nn.Linear class was painful as it only supports in,out formats, hence we used nn.Parameters
        # we also start at 1/k which is the mean and is what is used by the paper to see if allowing optimization here improves loss

        # it gets us a (18, 32, 1024) shaped tensor x2 for the two frames we take as input and that we need to bottlneck

        # so we will be working with a (2,18,32,1024) that we want to scale down to (9, 16, 32)
        #(18, 32, 1024)
        # in similar fashion to what we did earlier
        self.bottleneck = nn.Conv3d(self.n_embd, self.C, kernel_size=2, stride=2)
         # times 8 since we basically divide by 1/2 * 1/2 * 1/2 the other dims, such that 8/8 = 1

    def _bottleneck(self, t):
        T, H, W, emb = t.shape
        output = t # we don't mind sharing pointers since the input is just inner state before we reach the latent space
        # I wish I could draw this, but here, we basically grab patches of 2x2x2 over the 3 first dimensions 
        # which lands us with 8 1024 element vectors per patch, which we each element wise multiply to a 1024 x C matrix 
        # to reduce those patch 'tokens' into the channel space we want, the challenging part being how to implement this mental model 
        output = t 
        # here, using .view to either target the right format right away or to try for first -1, 2, 2, 2, emb chunk the tensor doesn't work
        # this is highkey super complex to implement,
        # I had to this time give a look to the GH repo showing the implementation of the rae encoder (https://github.com/mira-wm/mira/blob/main/src/mira/codec/rae_encoder.py)
        # and they seem to use a Conv3D whose documentation is quite confusing but Claude nicely summarized for my specific use
        # Conv3D being a 3d convolutional layer, as we have encountered before in CNNs but this time allowing to control the step and size of our convolutions
        # here nonetheless, Conv3d expects to see the channels first which means we have to somehow properly reshuffle our input tensor 
        # to ensure the semantics fit and we don't just change the view and juggle everything out of place for the sake of .view

        # one helpful way to visualize how to move this I found is that if we imagine a 18 x 32 matrix holding 1024 vectors and try to get the 
        # [17,31, :]'th vector, it is the same thing as having 1024 [17,32] grids each holding a single scalar, and this time iterating over
        # depth as in : [:, 17, 31], we get the exact same result. and here we just need to put it in that format accounting for the extra dimention that is time
        # and since the first dimension here for Conv3d is already time, we are gucci I believe

        output = output.permute(3, 0, 1, 2) # here permutating dim 0 and 3 does the rotation I tried to describe which leaves us with 1024 18*32 tensors x Time
        output = self.bottleneck(output) # applying trainable convolutional network to merge the 3d - chunks into a single value
        output = output.permute (1,2,3,0) # permuting everything back into order
        return output

    def forward(self, frames):
        # assuming we get a frames tensor of shape [T, C, W, H] as input already pre-formated with H and W being respectively 288 and 512
        T, C, H ,W = frames.shape
        with torch.no_grad():
            features = self.dino(pixel_values=frames, output_hidden_states=True) # getting all hidden state
            # features.hidden_state is a tuple of [T, 581, 1024]
            selected_features = [features.hidden_states[i+1] for i in self.layers] # all the layers we want for layer mixing
            last_feature = features.last_hidden_state
            # an array of (T, 581, 1024) tensor with 5 extra tokens we need to clean off
            # that all sit at the beginning so easy to clean off
            selected_features = torch.stack([f[:,5:,:] for f in selected_features]) # stack to concat the array into a single tensor
            selected_features = selected_features.view(len(self.layers), T, self.H//16, self.W//16, self.n_embd) #viewing it accordingly
        # now we can mix em up
        mixed = torch.zeros_like(selected_features[0])
        for f,w in zip(selected_features, self.mix):
            mixed+= f * w #stacking up the dim=0 dim through additiong
        # we end up with a mixed in shape (T, H//16, W//16, self.n_embd)
        # we can now simply bottleneck it as we defined to get it to the latent space
        latent_space = self._bottleneck(mixed) # finally

        return latent_space

/Users/achrafbayi/Desktop/MIRA/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<img src="./views/decoderspec.png" width="1100px">
<img src = "./views/DecoderHyperparams.png" width = "1100px">

In [ ]:
# now that we have the encoder, if we want to train, we need to build the decoder and later pre-process the data and I guess import 
# some val shards too to really see how good of a loss we will be able to get 

import math

class Decoder(nn.Module):
   def __init__(self, T=20, H = 9, W= 16, C=32, n_embd=1152):
      super().__init__()
      # this time, input is of the shape (T//2, H//2, W//2, C) (unchanged from the encoder) and we want to bring it back to 2x the dimn on T,W,W and n_embd on C
      # the first step is the unpatchification which is similar to the patchification we've done earlier,
      # we need to understand that we want to scale back C -> 1152, W//2 -> W, H//2 -> H and T//2 -> T 
      # here is how we proceed to do so, scaling C is trivial, we howevere here want to scale it to 4*1024 = 4096 through good ole matrix multiplication
      # which will hand us back (T//2,H//2,W//2,4096), and we want to view those 4608 vectors as 2 x 2 matrices of each 1152 elements which will scale 
      # H and W in both dimension by 2
      self.T = T
      self.H = H
      self.W = W
      self.C = C
      self.n_embd = n_embd
      self.spatial_upsample = nn.Linear(C, n_embd*4)
      # ourput will be in the following format : (T, 2H, 2W, n_embd) (i.e 10, 18, 32 , 1152)
      self.vit = ViT() # default hardcoded values are correct for the paper's hyperparams for now


   def _spatial_upsample(self, t):
      out = t
      out = self.spatial_upsample(out) # growing back the last dimension
      # doing the view operation we mentioned above to have the H x W matrix go from storing 4608 vectors to 2 x 2 sub matrices 
      # of 1024 elements each in its cells
      out = out.view(self.T, self.H, self.W, 2, 2, self.n_embd)
      # now here, the permuts -> reshape work to un-nest that 2x2 can get pretty trick and was hard to understand at first glance seeing how 
      # shitty the PyTorch documentation can sometimes be ( this one sucks : https://docs.pytorch.org/docs/2.14/generated/torch.permute.html)
      # but the one for reshape is actually useful here (https://docs.pytorch.org/docs/2.14/generated/torch.reshape.html)
      # what we will try to achieve is to just permute (i.e swap the stride layout according to Claude) to get a tensor of shape : 
      # (T, ((H, 2), (2, W)), n_embd) in which we will be working on the sub tensors : ((H, 2), (2, W)), visualizing those as a H x 2 matrix 
      # of 2 x W sub-matrices helps to see how the shape is conserved if ever layed down on paper, and reshape here will simply allow us to in the
      # following order : 
      # 1) : turn the 2 x W matrix into a 2W long vector (which is the original goal of 2x-ing W)
      # 2): now that we have a H x 2 matrix of (2W) long vectors, we will also reshape (i.e append the 2nd col downwards to the 1st)
      #     the H x 2 into a 2H long column of 2W long rows which is exactly a 2H x 2W matrix of 1152 scalar vectors which is 
      #     what we initially seeked to do, let's implement that :
      out = torch.permute(out, (0,1,3,2,4,5)) # doing the permutations to get a ((H, 2), (W, 2)) in the middle
      out = out.reshape(self.T, self.H, 2, self.W*2, self.n_embd) # one cool thing about torch.reshape is that we don't need to isolate the 
      # sub matrix we want to flatten, we just specify the new shape, and it AUTOMATICALLY grabs the elements from the axis to the right of
      # the axis we want to 'flatten/reshape' to do so which is why we got it into that shape through permute
      # now we just do step 2 and we're gucci on all dims but T
      out = out.reshape(self.T, self.H*2, self.W*2, self.n_embd)

      return out

   def forward(self, x):
      #x should be of shape (20, 9, 16, 32), we will first upscale it as seen in the _spatial_upsample
      out = _spatial_upsample(x)
      # out should now be of (20, 18, 32, 1152) shape which we will put through our built ViT
      out = self.vit(out)
      # now space and time attentioned, should be still of (20, 18, 32, 1152) shape, we now just need to upsample the time dimension and we should be done








# we need to build our own implementation of a space-time ViT whose hyperparameters are specified in Table 9


class ViT(nn.Module): # the paper specifies in table 9 a depth of 28 so 28 (attention->mlp) blocks stacked, let's just specify the blocks
   def __init__(self, depth=28, T=20, H=18, W=32, n_embd=1152):
      super().__init__()
      self.blocks = nn.ModuleList([Block() for _ in range(depth)]) # instantiating depth-long array of blocks
      self.positional_embeddings = nn.Parameter(torch.randn((T,H,W,n_embd)))*0.02 # learned positional embeddings, applied once  before as specified in the transformer architecture
      #scaling down the positional embeddings as seen in GPT-2 

   def forward(self, x):
      # once again, with (T, H, W, n_embd) shaped input, we'll sequentially apply the blocks
      out = x + self.positional_embeddings
      for block in self.blocks:
         out = block(out)
      return out


class Block(nn.Module):
   def __init__(self):
      super().__init__()
      self.attention = Attention() # we hardcoded the paper's default values correctley so no need to specify anything for now unless we would later like to change a hyperparameter
      self.mlp = MLP()

   def forward(self,x): # here again, input should be of format (T,H,W,n_embd)
      out = self.attention(x)
      out = self.mlp(out) 
      # btw all pre-norm layer normalization and residual connections are implemented within the attention and mlp blocks so no need to do anything here
      return out



class MLP(nn.Module): 
   def __init__(self, n_embd=1152, mlp_mult=4):
         super().__init__()
         # input comes as (T,H,W,n_embd) and we work with a 4x multiplier on n_embd
         self.n_embd = n_embd
         self.mlp_mult = mlp_mult
         self.ln = nn.LayerNorm(n_embd) # pre-norm ln learned gammas and betas
         self.layer1 = nn.Linear(n_embd, n_embd*mlp_mult) # scaling up to the MLP's dim
         self.non_linearity = nn.GELU() # non-linearity
         self.layer2 = nn.Linear(n_embd*mlp_mult, n_embd) # scaling back down to the embd's dim

   def forward(self, x):
      out = self.ln(x) # pre-norm
      out = self.layer1(out)
      out = self.non_linearity(out) #GELU
      out = self.layer2(out)
      out = out + x # residual connection
      return out

# works with (T, H, W n_embd) inputs and returns (T, H, W, n_embd) with space and time attention applied sequentially
class Attention(nn.Module):
   def __init__(self, n_head=16, n_embd=1152):
      super().__init__()
      self.n_embd=n_embd
      self.head_dim = n_embd//n_head
      self.space_heads = nn.ModuleList([SpaceAttentionHead() for _ in range(n_head)])
      self.time_heads = nn.ModuleList([TimeAttentionHead() for _ in range(n_head)])
      self.space_ln = nn.LayerNorm(n_embd)
      self.time_ln = nn.LayerNorm(n_embd) # since we'll work with the output of the 1st space attention layer
      self.space_mixing = nn.Linear(n_embd, n_embd)
      self.time_mixing = nn.Linear(n_embd, n_embd) # linear layers we'll use to mix the concatenations 

   def forward(self, x):
      # here x is of dim (T, H, W, n_embd)
      # we need first to apply spatial attention to all frames and then time attention too
      T, H, W, emb = x.shape
      out = self.space_ln(x) # pre-norm layer norm
      out = torch.concat([h(out) for h in self.space_heads], dim=-1).reshape(T,H,W,self.n_embd) # applying space attention, we also let a residual connection in
      # reshaping into 2d output for us to have coherent space and time attention head code and to be consistent
      out = self.space_mixing(out) #mixing
      out = out + x #residual connection for space attention (2d)
      newx = out # output of the space attention is the new input for the time attention
      out = self.time_ln(newx) # 2nd pre-norm layernorm
      out = torch.concat([h(out) for h in self.time_heads], dim=-1).reshape(T,H,W,self.n_embd) # stacking the columns
      out = self.time_mixing(out)
      out = out + newx # residual connection for time attention
      return out # simple this time


class SpaceAttentionHead(nn.Module): # not a join space-time attention, we are applying two reqular attention QK sequentially rather than a 3 dimensional one
   def __init__(self, H=18, W=32 , n_embd=1152, n_head=16, T=20):
      super().__init__()
      self.T = T
      self.H = H
      self.W = W
      self.n_embd = n_embd
      head_dim = n_embd//n_head
      #(T, H, W, n_embd) here is what is fed to us, we want to sequentially apply space attention (self) and later time attention (causal)
      # we consume frame by frame, hence working in ( H, W, n_embd) work space
      # we'll hence move into (H*W, n_embd) token space as specified in the ViT An image is a 16x16 word space as our token sequence
      # through a .reshape() on the input tensor hence we'll just need a (H*W, n_embd)-long learned matrix for Q, K and V
      self.wQ = nn.Linear(n_embd, head_dim)
      self.wK = nn.Linear(n_embd, head_dim) # @ matmul dot product to get the QK (H*W, H*W) shape
      self.wV = nn.Linear(n_embd, head_dim) # values to be multiplied to the layernormed, softmaxed attention through matmul again
      # all learned

      # so the outputed Q,K,V will be of shape :
      # Q.shape = (H*W, n_embd)
      # K.shape = (H*W, n_embd) we will .transpose to run the dot products properly
      # QK.shape = (H*W, H*W), good
      # V.shape = (H*W, n_embd//16)

      # finally the layer normalization learned gamma and betas 
      self.q_ln = nn.LayerNorm(head_dim)
      self.k_ln = nn.LayerNorm(head_dim)
   def forward(self, x):
      # x is of shape (H,W,n_embd)

      x = x.reshape(self.T, self.H*self.W, self.n_embd) # flatening the tokens (576 tokens)
      Q = self.wQ(x)
      K = self.wK(x)
      V = self.wV(x)
      # we now apply our layer norms real quick 
      Q = self.q_ln(Q)
      K = self.k_ln(K)
      dnorm = 1/(math.sqrt(K.shape[-1])) #dimentionality of K
      QK = Q @ K.transpose(-2,-1) # using permute to transpose to run the dot products
      QKnormed = QK * dnorm
      softmaxedQK = torch.softmax(QKnormed, dim=-1) # on the columns
      out = softmaxedQK @ V
      return out

class TimeAttentionHead(nn.Module):
   def __init__(self, n_head=16, n_embd=1152, T=20, H=18, W=32):
      super().__init__()
       # so here, again, we'll be working with (T, H, W, n_embd) tensor
      self.emb_head = n_embd//n_head
      self.wQ = nn.Linear(n_embd, self.emb_head)
      self.wK = nn.Linear(n_embd, self.emb_head)
      self.wV = nn.Linear(n_embd, self.emb_head)
      self.dk = 1/(math.sqrt(self.emb_head)) # dimensionality of the keys
      # we also need our layer norm layers for the QK normalization applies in layer norm
      self.ln_q= nn.LayerNorm(self.emb_head)
      self.ln_k= nn.LayerNorm(self.emb_head)

   


   def forward(self, x):
      # we'll first squash it down to 
      # (T, HxW, n_embd) tokens, and we'll hence want to compute attention this time on all the previous
      # mappings of that token in the other frame, to make that easier, we may want to try to reshape the input
      # to something like (H*W, T, n_embd) which can be seen as a matrix who'se rows represnt a specific position token on the image
      # and column value its time embedding, hence we may want to apply attention over itself and all previous ones which would indeed
      # be a lower-triangular matrix
      # let's first

      T,H,W,n_embd = x.shape

      x = x.reshape(T, H*W, n_embd) # automatically picks the right column to H which is W so it works here natively
      # now we want to permute
      x = x.permute(1,0,2) # going to (H*W, T, n_embd) (i.e, [frame1, frame2, frame3,....]) each frame being made of N*W tokens of embedding n_embd
      # and now we may want to generate our 
      Q = self.wQ(x)
      K= self.wK(x)
      V = self.wV(x)
      # Q.shape, K.shape, V.shape = (H*W, T, head_emb)
      # QK norm
      Q = self.ln_q(Q)
      K = self.ln_k(K)
      QK = Q @ K.permute(0, 2, 1) #pairwise matmul
      QK = QK * self.dk
      # now we need to mask
      mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device = x.device)) # of same shape as QK
      QK = QK.masked_fill(~mask, float('-inf')) # btw ~ is the bitwise not operator meaning on NOT True (bool True) it replaced by -inf
      out = torch.softmax(QK, dim=-1) # on the columns
      out = out @ V #(H*W, T, head_emb) shape through tensor-wise mat-mul
      out = out.permute(1,0,2) # going back to frames first
      return out








        
    